# Tutorial 2: Load and Explore a SAHA CosMx Sample

This notebook walks through:
1. Finding a sample of interest via metadata
2. Reading the cell table of its **SpatialData** object (only the table is fetched)
3. Basic QC and cell type visualization
4. Spatial plotting of gene expression

The release format is SpatialData (OME-NGFF / Zarr). Each store holds the
measurement table, cell polygons, transcript points and the morphology image
together; the table is an AnnData, so scanpy and squidpy work as usual.


## Setup

In [ ]:
# !pip install scanpy squidpy s3fs fsspec zarr anndata matplotlib seaborn

In [ ]:
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False)

import os

# One setting picks the data source; every path below is built from DATA_ROOT.
#   "s3://saha-public"     reviewer copy on Wasabi: public read, no credentials (default)
#   "s3://saha-open-data"  AWS Open Data bucket, once it is live
#   "/path/to/saha"        a local copy with the same layout: metadata/, processed/<record>/
DATA_ROOT = os.environ.get("SAHA_DATA_ROOT", "s3://saha-public")
if DATA_ROOT.startswith("s3://saha-public"):
    S3_OPTS = {"anon": True, "client_kwargs": {"endpoint_url": "https://s3.wasabisys.com"}}
elif DATA_ROOT.startswith("s3://"):
    S3_OPTS = {"anon": True}
else:
    S3_OPTS = {}
META = f"{DATA_ROOT}/metadata"

import fsspec, zarr
import anndata as ad

def object_path(sample_id):
    """The store for one acquisition: processed/<record>/<sample_id>.zarr.
    (25 Xenium stores carry an _RNA suffix, so both names are accepted.)"""
    fs, root = fsspec.core.url_to_fs(DATA_ROOT, **S3_OPTS)
    hits = [h for h in fs.glob(f"{root}/processed/*/{sample_id}*.zarr")
            if h.rsplit("/", 1)[-1] in (f"{sample_id}.zarr", f"{sample_id}_RNA.zarr")]
    assert len(hits) == 1, f"{sample_id}: {hits}"
    return fs.unstrip_protocol(hits[0]) if DATA_ROOT.startswith("s3://") else hits[0]

def read_table(sample_id):
    """The cell table (an AnnData) of one object, read straight from the store.
    Only the table is fetched; images, polygons and transcripts stay where they are."""
    g = zarr.open_group(object_path(sample_id), mode="r", storage_options=S3_OPTS or None)
    adata = ad.io.read_elem(g["tables/table"])
    # 25 Xenium tables keep scaled values in X and the counts in layers["raw"];
    # put counts in X so every object starts from the same place.
    if "raw" in adata.layers:
        adata.X = adata.layers["raw"]
    return adata

# Cell types: `cell_type_broad` is the one vocabulary shared by every object.
LABEL = "cell_type_broad"

# For the full SpatialData object (images, polygons, transcripts), copy the store
# first; spatialdata does not open a remote store reliably. For example:
#   aws s3 sync --no-sign-request --endpoint-url https://s3.wasabisys.com \
#       s3://saha-public/processed/<record>/<store>.zarr ./<store>.zarr
#   sdata = spatialdata.read_zarr("./<store>.zarr")


## Step 1: Find a sample

In [ ]:
samples = pd.read_parquet(f"{META}/samples.parquet", storage_options=S3_OPTS or None)

# Colon on CosMx RNA (the release has colon on the 1K panel only). Smallest first,
# so the table download stays at a few MB; take a larger one for real work.
candidates = samples.query(
    "organ == 'colon' and platform == 'cosmx' and assay_type == 'rna'"
).sort_values("n_cells")

print(f"Found {len(candidates)} candidates")
candidates[["sample_id", "run_id", "panel_name", "n_cells", "n_fovs"]].head(5)

In [ ]:
row = candidates.iloc[0]
print(f"Selected: {row['sample_id']}")
print(f"  organ:   {row['organ']}")
print(f"  n_cells: {row['n_cells']:,}")
print(f"  n_fovs:  {row['n_fovs']}")
print(f"  store:   {object_path(row['sample_id'])}")

## Step 2: Load the cell table (AnnData)

In [ ]:
# The release ships SpatialData (OME-NGFF / Zarr). The measurement table inside
# is an ordinary AnnData, so everything below is plain scanpy/squidpy.
adata = read_table(row["sample_id"])
print(adata)

In [ ]:
print(adata.obs.columns.tolist())

# Donor identity is per CELL, not per sample -- these slides are tissue
# microarrays carrying up to 28 cores:
if "donor_core" in adata.obs.columns:
    print("\ndonor cores in this acquisition:",
          adata.obs["donor_core"].nunique())


## Step 3: Basic QC

In [ ]:
sc.pp.calculate_qc_metrics(
    adata, percent_top=None, log1p=False, inplace=True
)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(adata.obs["total_counts"], bins=80, color="steelblue", edgecolor="none")
axes[0].set_xlabel("Total counts per cell")
axes[0].set_ylabel("# cells")
axes[0].set_title("Transcript counts")

axes[1].hist(adata.obs["n_genes_by_counts"], bins=80, color="coral", edgecolor="none")
axes[1].set_xlabel("Genes detected per cell")
axes[1].set_title("Genes per cell")

# Negative controls (if available)
neg_genes = [g for g in adata.var_names if g.startswith("NegPrb")]
if neg_genes:
    neg_counts = adata[:, neg_genes].X.sum(axis=1)
    axes[2].hist(neg_counts, bins=50, color="gray", edgecolor="none")
    axes[2].set_xlabel("Negative probe counts per cell")
    axes[2].set_title("Negative controls")
else:
    axes[2].text(0.5, 0.5, "No neg. probes", ha="center", va="center",
                  transform=axes[2].transAxes)

plt.tight_layout()
plt.show()

print(f"Median counts:        {adata.obs['total_counts'].median():.0f}")
print(f"Median genes/cell:    {adata.obs['n_genes_by_counts'].median():.0f}")

## Step 4: Cell type UMAP

In [ ]:
# If UMAP is already in adata.obsm, just plot it
if "X_umap" in adata.obsm:
    sc.pl.umap(
        adata,
        color=LABEL,
        legend_loc="on data",
        legend_fontsize=8,
        title=row["sample_id"],
        frameon=False,
    )
else:
    print("UMAP not precomputed — running from scratch ...")
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.pca(adata, n_comps=30)
    sc.pp.neighbors(adata, n_neighbors=15)
    sc.tl.umap(adata)
    sc.pl.umap(adata, color=LABEL, frameon=False)

## Step 5: Spatial cell type map

In [ ]:
# Spatial cell type map — the hallmark of spatial omics
# (mapping a categorical column to RGB tuples by hand fails in recent pandas, so let scanpy color it)
if "spatial" in adata.obsm:
    fig, ax = plt.subplots(figsize=(12, 9), dpi=100)
    sc.pl.embedding(adata, basis="spatial", color=LABEL, size=2, ax=ax, show=False, frameon=False,
                    title=f"{row['sample_id']} — cell types in tissue")
    ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()
else:
    print("No spatial coordinates found in this sample.")

## Step 6: Spatial gene expression

In [ ]:
# Pick a few marker genes to visualize spatially
marker_genes = [g for g in ["EPCAM", "CD3E", "CD68", "MKI67", "COL1A1"]
                if g in adata.var_names]

if marker_genes and "spatial" in adata.obsm:
    sq.pl.spatial_scatter(
        adata,
        color=marker_genes,
        shape=None,
        size=2,
        img=False,
        ncols=len(marker_genes),
        figsize=(4 * len(marker_genes), 4),
    )
else:
    print(f"Marker genes in panel: {marker_genes}")

## Step 7: Cell type composition

In [ ]:
ct_counts = adata.obs[LABEL].value_counts()

fig, ax = plt.subplots(figsize=(6, 5))
ct_counts.plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("# cells")
ax.set_title(f"Cell type composition\n{row['sample_id']}")
plt.tight_layout()
plt.show()

print(ct_counts.to_string())

## Step 8: Neighborhood enrichment (spatial statistics)

In [ ]:
if "spatial" in adata.obsm and LABEL in adata.obs.columns:
    # Build spatial neighbors graph
    nh = adata[adata.obs[LABEL].notna()].copy()
    nh.obs[LABEL] = nh.obs[LABEL].cat.remove_unused_categories()
    sq.gr.spatial_neighbors(nh, coord_type="generic", delaunay=True)

    # Neighborhood enrichment score
    sq.gr.nhood_enrichment(nh, cluster_key=LABEL)
    sq.pl.nhood_enrichment(nh, cluster_key=LABEL,
                            title="Neighborhood enrichment",
                            figsize=(8, 6))

## Next steps

- **Tutorial 03** — Cross-platform comparison (CosMx + Xenium) of the same organ
- Load raw CosMx flat files using `spatialdata-io` for FOV-level analysis
- See [FILE_FORMATS.md](../docs/FILE_FORMATS.md) for raw data format details